## Import available `cdapython` functions

In [ ]:
try:
  from itables import(
     init_notebook_mode, show 
    ) 
except:
  !pip install -r https://raw.githubusercontent.com/CancerDataAggregator/Community-Notebooks/main/requirements.txt


import numpy as np
import pandas as pd
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt

opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0

from cdapython import *

In [ ]:
from cdapython import *


To see help pages while you work type:

```
help(function)
```

example:

```
help(get_subject_data)
```

## Get a list of searchable CDA tables

In [ ]:
tables()

## Explore CDA tables' columns in detail

In [ ]:
columns()

## See what values are populated in a given column

In [ ]:
column_values( 'diagnosis' )

## Search without knowing a column name at all

Every `summarize_` and `get_` function below also accepts one or more bare search terms, searched across every column at once -- useful as a first pass before you know any column names:

In [ ]:
summarize_subjects('kidney')

## Fetch subject row summary information for a column value

In [ ]:
summarize_subjects( match_all = 'anatomic_site = kid*')

## Fetch file row summary information for a column value

`summarize_files()` works just like `summarize_subjects()`, but profiles the `file` table instead:

In [ ]:
summarize_files( match_all = 'anatomic_site = kid*')

## Fetch subject rows for a column value

In [ ]:
get_subject_data( match_all = 'subject_id = TCGA.TCGA-04-1369')

## Fetch file rows for a column value

`get_file_data()` works just like `get_subject_data()`, but returns full rows from the `file` table instead:

In [ ]:
get_file_data( match_all = 'subject_id = TCGA.TCGA-04-1369')

## Restrict a search to specific data sources

Any of the functions above accept `data_source` to restrict to one or more of CDA's six sources: `'GDC'`, `'PDC'`, `'IDC'`, `'GC'`, `'ICDC'`, `'CTDC'`. (`column_values()` is the one exception -- it only accepts a single source, not a list.)

In [ ]:
get_subject_data( match_all = 'anatomic_site = kid*', data_source = ['GDC', 'IDC'])

## Pull in columns from another table

`add_columns` adds columns from a second table to your results. A single column name adds just that column; a table name followed by `.*` (e.g. `'file.*'`) adds every column from that table:

In [ ]:
get_subject_data( match_all = 'anatomic_site = kid*', add_columns = 'file.*')

## Keep related values paired up

By default, values added via `add_columns` get mashed into separate lists per subject, so you can't tell which values actually belong together. `collate_results=True` pairs them up instead:

In [ ]:
kidney_collated = get_subject_data( match_all = 'anatomic_site = kid*', add_columns = 'file.*', collate_results = True)
kidney_collated

## Flatten a collated result

`collate_results=True` gives you a nested column (here, `file_data`) with a small table inside each row. `expand_subject_results()` flattens that into a plain table, one row per file instead of one row per subject:

In [ ]:
expand_subject_results(kidney_collated, 'file_data')

(`expand_file_results()` is the file-table equivalent, for use on results from `get_file_data()`.)

## See why a harmonized value matched

`add_extras` shows the real underlying vocabulary behind a harmonized match, without changing which rows matched. It's most useful on `anatomic_site` and `disease`, the two fields backed by a real ontology (UBERON and ICD-O-3):

In [ ]:
summarize_subjects('kidney', add_extras = ['all'])

## Combine two separate searches

If you need an "AND" across two different values of the *same* column -- like two different `file_type`s, where no single file can be both -- `match_all` can't express that in one call. Run two searches and intersect them instead:

In [ ]:
ct_subjects = get_subject_data(match_all=['anatomic_site = kid*', 'file_type = CT Image Storage'])
mutation_subjects = get_subject_data(match_all=['anatomic_site = kid*', 'file_type = Annotated Somatic Mutation'])
intersect_subject_results(ct_subjects, mutation_subjects)

(`intersect_file_results()` is the file-table equivalent, for use on two `get_file_data()` results.)

## Check your own list of IDs against CDA

`match_from_file` lets you check a TSV of IDs you already have against CDA, instead of retyping each one into a filter. If your file has a data center's own native ID (e.g. a GDC case UUID), match against `'upstream_id'`. If it has CDA's own subject IDs, match against `'subject_id'` instead.

This example pulls a sample `cases.tsv` straight from the Community-Notebooks repo, so you don't need to download or upload anything yourself:

In [ ]:
!curl -sL -o cases.tsv https://raw.githubusercontent.com/CancerDataAggregator/Community-Notebooks/main/Tutorials/cases.tsv

In [ ]:
get_subject_data(
    match_from_file={
        'input_file': 'cases.tsv',
        'input_column': 'id',
        'cda_column_to_match': 'upstream_id'
    }
)